In [1]:
from py2neo import Graph,Node,Relationship
import json
from tqdm import tqdm
from pprint import pprint


In [2]:
graph = Graph("bolt://localhost:7687",user = "neo4j", password = "Ljp13249158398@")
print("连接成功")

连接成功


In [ ]:
event_type_file = '/data/rag_full_stack_course_notebooks/notebook/data/iree.json'
event_type_data = []
with open(event_type_file, 'r', encoding='utf-8') as file:
    # 尝试读取第一行来判断格式
    first_line = file.readline().strip()
    if not first_line:
        print("文件为空")
    else:
        # 重置文件指针到开头
        file.seek(0)
        
        # 简单判断：JSON 数组
        if first_line.startswith('['):
            try:
                event_type_data = json.load(file)
                if not isinstance(event_type_data, list):
                    event_type_data = [event_type_data]
            except json.JSONDecodeError as e:
                print(f"标准 JSON 解析失败: {e}")
        else:
            # 按 JSON Lines 处理
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    event_type_data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"跳过第 {line_num} 行，JSON 格式错误: {e}")

In [4]:
pprint(event_type_data[90], sort_dicts=False)

{'title': '国脉科技(002093.SZ)：股东林惠榕解除质押2888.17万股',
 'content': '国脉科技(002093.SZ)公布，公司近日接到股东林惠榕书面告知，获悉林惠榕已将其持有并质押给厦门国际银行的2888.1701万股办理解除质押，占其所持股份的13.09%。截至公告日，林惠榕及其其一致行动人合计持有公司股份5.5044亿股，占公司总股本的54.63%；累计质押公司股份1.8842亿股，占其合计所持股份的34.23%，占公司总股本的18.70%。 '
            '相关股票 国脉科技 SZ 002093 9.91 +0.31 +3.23%',
 'label': []}


In [5]:
invest_file = '/data/rag_full_stack_course_notebooks/notebook/data/invest-on-invent-KG.json'
with open(invest_file, 'r', encoding='utf-8') as file:
    invest_data = json.load(file)

In [7]:
invest_data['@graph'][6382]

{'@id': '6382',
 '@type': 'company',
 'name': '广州中设机器人智能装备股份有限公司',
 'alterNames': ['中设机器人智能装备',
  '中设机器人智能装备公司',
  '广州中设机器人智能装备',
  '广州市中设机器人智能装备',
  '广州中设机器人智能装备公司',
  '广州市中设机器人智能装备公司'],
 'establishDate': '2008-07-23',
 'address': '广州市黄埔区云埔工业区方达路6号101房',
 'relationship': {'applyPatent': []}}

In [8]:
obj_by_id = {}

for ele in invest_data["@graph"]:
    obj_by_id[ele["@id"]] = ele

In [9]:
ignore_names = ['电器', '600万股', '公司', '135万元', '2018年', '新产业']
even_type_company_names = {}
for data in event_type_data:
    for label in data['label']:
        if 'arguments' in label:
            for arg in label['arguments']:
                for key, value in arg.items():
                    if key == "主体":
                        if value in ignore_names:
                            continue
                        even_type_company_names[value] = 1

In [10]:
list(even_type_company_names.keys())[:10]

['小米集团',
 '小米',
 '格力电器',
 '波音公司',
 '波音',
 'GoPro',
 '得邦照明股份有限公司',
 '得邦照明',
 '横店集团得邦工程塑料有限公司',
 '汉邦高科']

In [11]:
invest_company_names = []

for ele in invest_data["@graph"]:
    if ele["@type"] == "company":
        invest_company_names.append(ele["name"])
invest_company_names[:10]

['广州博鳌纵横网络科技有限公司',
 '福建博思软件股份有限公司',
 '深圳优地科技有限公司',
 '杭州群核信息技术有限公司',
 '昆山希盟自动化科技有限公司',
 '北京小爱智能科技有限公司',
 '奇安信科技集团股份有限公司',
 '北京大米科技有限公司',
 '北京诸葛找房信息技术有限公司',
 '苏州奥易克斯汽车电子有限公司']

In [12]:
even_type_company_dict = {}
for company in even_type_company_names.keys():
    for iv_cname in invest_company_names:
        if company in iv_cname:
            even_type_company_dict.setdefault(company, []).append(iv_cname)

In [13]:
even_type_company_dict['小米']

['小米科技有限责任公司']

In [14]:
for data in tqdm(event_type_data):
    title = data['title']
    content = data['content']
    
    for label in data['label']:
        event_type = label['event_type']
        event_type_node = Node("EventType", name=event_type)
        graph.merge(event_type_node, "EventType", "name")
        
        if 'arguments' in label:
            
            cname = ''
            tmp_args = {}
            for arg in label['arguments']:
                for key, value in arg.items():
                    if key == "主体":
                        cname = value
                    else:
                        tmp_args[key] = value
            if cname in even_type_company_dict:
                for company_name in even_type_company_dict[cname]:
                    company_node = Node("Company", name=company_name)
                    graph.merge(company_node, "Company", "name")
                    
                    relationship = Relationship(company_node, "HAPPEN", event_type_node)
                    for key, value in tmp_args.items():
                        relationship[key] = value
                    
                    relationship['title'] = title
                    relationship['content'] = content
                    graph.create(relationship)

100%|██████████| 7058/7058 [00:48<00:00, 146.86it/s]


In [ ]:
import json
import csv
import os
from tqdm import tqdm

# ====================== 加载数据（和你 notebook 完全一致） ======================
event_type_file = '/data/raggggg/员工制度rag/data/iree.json'
event_type_data = []
with open(event_type_file, 'r', encoding='utf-8') as file:
    for line_num, line in enumerate(file, 1):
        line = line.strip()
        if not line:
            continue
        try:
            event_type_data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"跳过第 {line_num} 行，JSON 格式错误: {e}")

invest_file = '/data/raggggg/员工制度rag/data/invest-on-invent-KG.json'
with open(invest_file, 'r', encoding='utf-8') as file:
    invest_data = json.load(file)

obj_by_id = {ele["@id"]: ele for ele in invest_data["@graph"]}

# 创建输出目录
os.makedirs('./import_csv', exist_ok=True)


print("生成 Company 节点 CSV...")
company_rows = []
seen_companies = set()

for ele in invest_data["@graph"]:
    if ele.get("@type") == "company":
        name = ele.get("name")
        if name in seen_companies:
            continue
        seen_companies.add(name)
        company_rows.append([
            name,
            ele.get("establishDate", ""),
            ele.get("address", "")
        ])

with open('./import_csv/companies.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['name:ID(Company)', 'establishDate', 'address'])
    writer.writerows(company_rows)

print("生成 Investor 节点 CSV...")
investor_rows = []
for ele in invest_data["@graph"]:
    if ele.get("@type") == "investor":
        investor_rows.append([ele.get("name")])

with open('./import_csv/investors.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['name:ID(Investor)'])
    writer.writerows(investor_rows)


print("生成 Patent 和 HAVE 关系 CSV...")
patent_rows = []
have_rows = []
seen_patents = set()

for ele in invest_data["@graph"]:
    if ele.get("@type") == "company":
        comp_name = ele.get("name")
        for patent in ele["relationship"].get("applyPatent", []):
            if patent['@id'] in obj_by_id:
                patent_name = obj_by_id[patent['@id']]['name']
                if patent_name not in seen_patents:
                    seen_patents.add(patent_name)
                    patent_rows.append([patent_name])
                have_rows.append([comp_name, patent_name])   # :START_ID :END_ID

with open('./import_csv/patents.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['name:ID(Patent)'])
    writer.writerows(patent_rows)

with open('./import_csv/have_rels.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([':START_ID(Company)', ':END_ID(Patent)'])
    writer.writerows(have_rows)


print("生成 INVEST 关系 CSV...")
invest_rows = []
for ele in invest_data["@graph"]:
    if ele.get("@type") == "investor":
        inv_name = ele.get("name")
        for invest in ele["relationship"].get("investCompany", []):
            if invest['@id'] in obj_by_id:
                comp_name = obj_by_id[invest['@id']].get("name")
                invest_rows.append([
                    inv_name,
                    comp_name,
                    invest.get("date", ""),
                    invest.get("round", "")
                ])

with open('./import_csv/invest_rels.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([':START_ID(Investor)', ':END_ID(Company)', 'date', 'round'])
    writer.writerows(invest_rows)


print("生成 EventType 和 HAPPEN 关系 CSV...")
eventtype_set = set()
happen_rows = []
ignore_names = ['电器', '600万股', '公司', '135万元', '2018年', '新产业']

for data in event_type_data:
    title = data.get('title', '')
    content = data.get('content', '')
    for label in data.get('label', []):
        event_type = label.get('event_type')
        if not event_type:
            continue
        eventtype_set.add(event_type)
        
        cname = ''
        tmp_args = {}
        for arg in label.get('arguments', []):
            for key, value in arg.items():
                if key == "主体":
                    cname = value
                else:
                    tmp_args[key] = value
        
        if cname and cname not in ignore_names:
            target_companies = even_type_company_dict.get(cname, [cname]) if 'even_type_company_dict' in globals() else [cname]
            for comp_name in target_companies:
                row = [comp_name, event_type, title, content]
                for k, v in tmp_args.items():
                    row.append(str(v)) 
                happen_rows.append(row)

with open('./import_csv/eventtypes.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['name:ID(EventType)'])
    writer.writerows([[et] for et in eventtype_set])

with open('./import_csv/happen_rels.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([':START_ID(Company)', ':END_ID(EventType)', 'title', 'content'])
    writer.writerows(happen_rows)

print("完成：./import_csv/")
print(f"Company: {len(company_rows)} 条 | Investor: {len(investor_rows)} 条 | Patent: {len(patent_rows)} 条")
print(f"INVEST: {len(invest_rows)} 条 | HAPPEN: {len(happen_rows)} 条")

In [ ]:
from neo4j import GraphDatabase
from tqdm import tqdm

driver = GraphDatabase.driver("bolt://127.0.0.1:7687", 
                              auth=("neo4j", "Ljp13249158398@"), 
                              database="neo4j")

BATCH_SIZE = 3000   # 比较安全的批次大小

print("开始重新导入事件关系（HAPPEN）...")

event_batch = []

for data in tqdm(event_type_data, desc="准备事件数据"):
    title = data.get('title', '')
    content = data.get('content', '')
    
    for label in data.get('label', []):
        event_type = label.get('event_type')
        if not event_type:
            continue
            
        cname = ''
        tmp_args = {}
        for arg in label.get('arguments', []):
            for key, value in arg.items():
                if key == "主体":
                    cname = value
                else:
                    tmp_args[key] = value
        
        # 严格使用你原来的映射逻辑
        if cname in even_type_company_dict:
            for company_name in even_type_company_dict[cname]:
                event_batch.append({
                    "company_name": company_name,
                    "event_type": event_type,
                    "title": title,
                    "content": content,
                    **tmp_args   # 保留时间、数值等额外属性
                })

# 批量导入
success_count = 0
with driver.session() as session:
    for i in tqdm(range(0, len(event_batch), BATCH_SIZE), desc="导入 HAPPEN 关系"):
        batch = event_batch[i:i + BATCH_SIZE]
        result = session.run("""
            UNWIND $batch AS row
            MATCH (c:Company {name: row.company_name})
            MATCH (e:EventType {name: row.event_type})
            MERGE (c)-[r:HAPPEN]->(e)
            SET r.title = row.title,
                r.content = row.content,
                r += row
            RETURN count(r) as imported
        """, batch=batch)
        
        imported = result.single()["imported"]
        success_count += imported

print(f"\ 事件关系导入完成{len(event_batch)} 条，事件导入 {success_count} 条")

In [ ]:
query = """
MATCH (company)-[r:HAPPEN]->(e:EventType {name: "业绩承诺未达标"})
RETURN company.name as name,r.title as title
"""

# 执行查询
results = graph.run(query)
for record in results:
    print(dict(record))

In [19]:
query = """
MATCH (c:Company)-[r:HAPPEN]->(e)
WHERE c.name CONTAINS "比亚迪"
RETURN c.name as name, r.title as title, e.name as even
"""
results = graph.run(query)
for record in results:
    print(dict(record))

{'name': '比亚迪股份有限公司', 'title': '比亚迪上半年新增借款138.7亿 连续三年新增借款超上年净资产20％', 'even': '债务增加'}
{'name': '深圳比亚迪电动汽车投资有限公司', 'title': '比亚迪上半年新增借款138.7亿 连续三年新增借款超上年净资产20％', 'even': '债务增加'}
{'name': '比亚迪股份有限公司', 'title': '方向错了？动力电池市场将再掀起磷酸铁锂热潮', 'even': '产品创新'}
{'name': '深圳比亚迪电动汽车投资有限公司', 'title': '方向错了？动力电池市场将再掀起磷酸铁锂热潮', 'even': '产品创新'}
{'name': '比亚迪股份有限公司', 'title': '车界：新能源车，成也补贴，败也补贴', 'even': '利润下滑'}
{'name': '深圳比亚迪电动汽车投资有限公司', 'title': '车界：新能源车，成也补贴，败也补贴', 'even': '利润下滑'}
{'name': '比亚迪股份有限公司', 'title': '比亚迪(01211.HK)与日野汽车合作研发纯电动商用车', 'even': '业务合作'}
{'name': '深圳比亚迪电动汽车投资有限公司', 'title': '比亚迪(01211.HK)与日野汽车合作研发纯电动商用车', 'even': '业务合作'}
{'name': '比亚迪股份有限公司', 'title': '好事连连', 'even': '引进战略投资'}
{'name': '深圳比亚迪电动汽车投资有限公司', 'title': '好事连连', 'even': '引进战略投资'}
{'name': '比亚迪股份有限公司', 'title': '比亚迪不单单是家车企，这4个副业个个有名，难怪能成领头羊！', 'even': '企业转型'}
{'name': '深圳比亚迪电动汽车投资有限公司', 'title': '比亚迪不单单是家车企，这4个副业个个有名，难怪能成领头羊！', 'even': '企业转型'}
